In [55]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, log_loss, mean_squared_error
from torch.utils.data import DataLoader
from tqdm import tqdm
from itertools import combinations
from sklearn.model_selection import train_test_split

import heapq
from random import randrange
from random import seed as set_seed
import numpy as np
from numba import njit, prange
from pandas.api.types import is_numeric_dtype

import numpy as np
import pandas as pd
import torch.utils.data

from scipy.sparse.linalg import svds

In [56]:
@njit
def split_top_continuous(tasks, priorities):
    """
    Sample a sequence of unique tasks of the highest priority ensuring that
    no task will have another instance with the priority level above the lowest priority in the sequence.
    Usecases: avoiding issues with "recommendations from future" when splitting test data by timestamp.
    """
    priority_queue = [(-max(priorities), len(priorities))]  # initialize typed
    priority_queue.pop()

    for idx, priority in enumerate(priorities):
        heapq.heappush(priority_queue, (-priority, idx))

    topseq = {}  # continuous sequence of top-priority tasks
    nonseq_idx = []  # top-priority tasks that interrupt continuous sequence

    unique_tasks = set(tasks)
    while unique_tasks:
        _, idx = heapq.heappop(priority_queue)
        task = tasks[idx]
        try:
            visited = topseq[task]
        except:
            unique_tasks.remove(task)
        else:
            nonseq_idx.append(visited)
        topseq[task] = idx

    topseq_idx = [idx for _, idx in topseq.items()]
    lowseq_idx = [idx for _, idx in priority_queue]  # all remaining tasks
    return topseq_idx, lowseq_idx, nonseq_idx


def to_numeric_array(series):
    if not is_numeric_dtype(series):
        if not hasattr(series, "cat"):
            series = series.astype("category")
        return series.cat.codes.values
    return series.values


# see polara

In [57]:
def earliest_last_out(data, userid="user_id", priority="timestamp", copy=False):
    """
    It helps avoiding "recommendations from future", when training set contains
    events that occur later than some events in the holdout and can therefore
    provide an oracle hint for the algorithm.
    """
    holdout_idx, observed_idx, future_idx = split_top_continuous(
        to_numeric_array(data[userid]), data[priority].values
    )

    observed = data.iloc[observed_idx]
    holdout = data.iloc[holdout_idx]
    future = data.iloc[future_idx]

    if copy:
        observed = observed.copy()
        holdout = holdout.copy()
        future = future.copy()

    return observed, holdout, future


# see polara

In [58]:
class MovieLens20MDataset(torch.utils.data.Dataset):
    """
    MovieLens 20M Dataset

    Data preparation
        treat samples with a rating less than 3 as negative samples

    :param dataset_path: MovieLens dataset path

    Reference:
        https://grouplens.org/datasets/movielens
    """

    def __init__(self, dataset_path, sep=",", engine="c", header="infer"):
        self.data = pd.read_csv(
            dataset_path, sep=sep, engine=engine, header=header
        ).to_numpy()[:, :4]
        self.items = (
            self.data[:, :2].astype(np.int32) - 1
        )  # -1 because ID begins from 1
        self.targets = self.__preprocess_target(self.data[:, 2]).astype(np.float32)
        self.field_dims = np.max(self.items, axis=0) + 1
        self.user_field_idx = np.array((0,), dtype=np.int64)
        self.item_field_idx = np.array((1,), dtype=np.int64)

    def __len__(self):
        return self.targets.shape[0]

    def __getitem__(self, index):
        return self.items[index], self.targets[index]

    def __preprocess_target(self, target):
        target[target <= 0] = 0
        target[target > 0] = 1
        return target


class MovieLens1MDataset(MovieLens20MDataset):
    """
    MovieLens 1M Dataset

    Data preparation
        treat samples with a rating less than 3 as negative samples

    :param dataset_path: MovieLens dataset path

    Reference:
        https://grouplens.org/datasets/movielens
    """

    def __init__(self, dataset_path):
        super().__init__(dataset_path, sep="::", engine="python", header=None)

In [59]:
dataset_path = "data/MovieLens1M/ratings.dat"
dataset = MovieLens1MDataset(dataset_path)

In [60]:
user_num = dataset.field_dims[0]
item_num = dataset.field_dims[1]
print("Number of users: ", user_num, ", Number of items: ", item_num)

Number of users:  6040 , Number of items:  3952


In [61]:
columns_name = ["user_id", "item_id", "rating", "timestamp"]
df = pd.DataFrame(dataset.data, columns=columns_name)
df

,user_id,item_id,rating,timestamp
0,1,1193,1,978300760
1,1,661,1,978302109
2,1,914,1,978301968
3,1,3408,1,978300275
4,1,2355,1,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,1,956704887
1000206,6040,562,1,956704746
1000207,6040,1096,1,956715648


In [62]:
df_sorted = df.sort_values(by="timestamp")

# Рассчитываем размер тестовой выборки (3%)
stab_size = int(len(df_sorted) * 0.9)
stab_size1 = int(len(df_sorted) * 0.008)


# Разделяем данные на train и test
stab_train = df_sorted.head(stab_size)
new_train_1 = stab_train.tail(stab_size1)
all_train = df_sorted.tail(len(df_sorted) - stab_size)
observed, holdout, future = earliest_last_out(all_train)

train = stab_train
train_new = pd.concat([new_train_1, observed], ignore_index=True)
train_old = train.head(stab_size - stab_size1)
test = holdout
test = test.sort_values(by="timestamp")
split_index = int(len(test) * 0.5)
valid = test.iloc[:split_index]
test = test.iloc[split_index:]
test_inds = test["user_id"].unique()
valid_inds = valid["user_id"].unique()
new_users = train_new["user_id"].unique()
valid_new = set(new_users) & set(valid_inds)
test_new = set(new_users) & set(test_inds)

In [63]:
print("Dataset statistics:")
print("Train length: ", len(train_old))
print("Train users: ", len(train_old["user_id"].unique()))
print("New Train length: ", len(train_new))
print("New Train users: ", len(train_new["user_id"].unique()))
print("Val length: ", len(valid))
print("Val users: ", len(valid["user_id"].unique()))
print("Test length: ", len(test))
print("Test users: ", len(test["user_id"].unique()))
print("Valid-new intersection: ", len(valid_new))
print("Test-new intersection: ", len(test_new))

Dataset statistics:
Train length:  892187
Train users:  5956
New Train length:  8212
New Train users:  181
Val length:  604
Val users:  604
Test length:  605
Test users:  605
Valid-new intersection:  103
Test-new intersection:  34


In [65]:
print(np.intersect1d(valid["user_id"].unique(), test["user_id"].unique()))

[]


In [11]:
from collections import defaultdict, Counter

train_user_history = defaultdict(list)
train_newuser_history = defaultdict(list)
val_user_history = defaultdict(list)
test_user_history = defaultdict(list)

In [42]:
def type_convers(data, mas):
    for _, row in tqdm(data.iterrows()):
        user_raw_id = row.user_id
        item_raw_id = row.item_id
        interaction_timestamp = row.timestamp
        mas[user_raw_id].append(
            {"item_id": item_raw_id, "timestamp": interaction_timestamp}
        )
    return mas

In [43]:
train_old

,user_id,item_id,rating,timestamp
1000138,6040,858,1,956703932
1000153,6040,2384,1,956703954
999873,6040,593,1,956703954
1000007,6040,1961,1,956703977
1000192,6040,2019,1,956703977
...,...,...,...,...
289962,1733,799,1,977808002
290549,1733,3727,1,977808038
290533,1733,3576,1,977808038
290202,1733,1321,1,977808038


In [44]:
train_user_history = type_convers(train_old, train_user_history)

892187it [03:19, 4474.47it/s]


In [45]:
train_newuser_history = type_convers(train_new, train_newuser_history)
val_user_history = type_convers(valid, val_user_history)
test_user_history = type_convers(test, test_user_history)

8212it [00:01, 4514.01it/s]
302it [00:00, 4013.97it/s]
907it [00:00, 4117.20it/s]


In [50]:
def tmp_transform(user_history, threshold=5):
    user_mapping = {}
    item_mapping = {}
    tmp_user_history = defaultdict(list)
    tmp_item_history = defaultdict(list)

    for user_id, history in tqdm(user_history.items()):
        processed_history = []

        for filtered_item in history:
            item_id = filtered_item["item_id"]
            item_timestamp = filtered_item["timestamp"]

            processed_item_id = item_mapping.get(item_id, len(item_mapping) + 1)
            item_mapping[item_id] = processed_item_id

            processed_history.append(
                {"item_id": processed_item_id, "timestamp": item_timestamp}
            )

        if len(processed_history) >= threshold:
            processed_user_id = user_mapping.get(user_id, len(user_mapping) + 1)
            user_mapping[user_id] = processed_user_id

            tmp_user_history[processed_user_id] = sorted(
                processed_history, key=lambda x: x["timestamp"]
            )

    return tmp_user_history

In [51]:
train_user_history = tmp_transform(train_user_history)
train_newuser_history = tmp_transform(train_newuser_history)
val_user_history = tmp_transform(val_user_history)
test_user_history = tmp_transform(test_user_history)

100%|█████████████████████████████████████| 907/907 [00:00<00:00, 219707.41it/s]


In [52]:
def train_samples_creator(user_history, threshold=5):
    samples = []
    for user_id, user_interactions in tqdm(user_history.items()):
        train_history = []

        for user_interaction in user_interactions:
            train_history.append(user_interaction)

        if len(train_history) >= threshold:  # remove cold-start users
            samples.append({"user_id": user_id, "history": train_history})
    return samples

In [53]:
def vt_samples_creator(user_history, train_history):
    samples = []
    for user_id, train_interactions in tqdm(train_history.items()):
        history = []
        for train_interaction in train_interactions:
            history.append(train_interaction)

        for user_interaction in user_history[user_id]:
            if len(history) >= 5:  # remove cold-start users
                test_samples.append(
                    {
                        "user_id": user_id,
                        "history": [x for x in history],
                        "next_interaction": user_interaction,
                    }
                )

    return samples

In [54]:
train_samples = train_samples_creator(train_user_history)
newtrain_samples = train_samples_creator(train_newuser_history, 1)
val_samples = vt_samples_creator(val_user_history, train_user_history)
test_samples = vt_samples_creator(test_user_history, train_user_history)

100%|████████████████████████████████████| 5951/5951 [00:00<00:00, 27460.38it/s]


In [ ]:
# train_samples = []
# newtrain_samples = []
# valid_samples = []
# test_samples = []

# for user_id, user_interactions in tqdm(user_history.items()):
#     train_history = []
#     history = []

#     for user_interaction in user_interactions:
#         if user_interaction['timestamp'] < fst_threshold: # train event
#             assert len(history) == 0 or user_interaction['timestamp'] >= history[-1]['timestamp']
#             train_history.append(user_interaction)
#         elif user_interaction['timestamp'] < snd_threshold: # valid event
#             assert user_interaction['timestamp'] >= fst_threshold
#             if len(history) >= 5:  # remove cold-start users
#                 valid_samples.append({
#                     'user_id': user_id,
#                     'history': [x for x in history],
#                     'next_interaction': user_interaction
#                 })
#         else:  # test event
#             assert user_interaction['timestamp'] >= snd_threshold
#             if len(history) >= 5:  # remove cold-start users
#                 test_samples.append({
#                     'user_id': user_id,
#                     'history': [x for x in history],
#                     'next_interaction': user_interaction
#                 })
#         history.append(user_interaction)

#     if len(train_history) >= 5:  # remove cold-start users
#         train_samples.append({
#             'user_id': user_id,
#             'history': train_history
#         })